# Experimento J02 - reutilización del decoder preentrenado de OWSM

## Pregunta experimental

> ¿Puede un encoder de sEEG ya entrenado con CTC alimentar directamente al decoder
> preentrenado de OWSM, sin volver a entrenar el encoder?

Este cuaderno separa dos preguntas que no deben mezclarse:

1. **Zero-shot (resultado principal):** encoder de sEEG congelado + decoder OWSM
   intacto, sin entrenamiento adicional. Mide compatibilidad directa entre los dos
   espacios latentes.
2. **Ajuste ligero (opcional):** el mismo encoder permanece congelado; se entrenan
   únicamente un puente residual `384 -> 384` y adaptadores LoRA en la atención
   cruzada del decoder. Mide si el conocimiento lingüístico del decoder puede
   recuperarse con una alineación barata.

El decoder recibe los estados ocultos `(T, 384)` del encoder. **No recibe los
`logits_ctc`** y no hay ninguna conversión desde las clases de caracteres o palabras
hacia las 50.002 clases de OWSM. El vocabulario BPE pertenece al decoder.

### Elección del encoder

El notebook busca, por este orden:

1. `B01_word_final`, si su entrenamiento ya terminó y guardó checkpoint;
2. el checkpoint de palabras `B3word` del barrido;
3. `G01_char_final`;
4. `H2`, el mejor encoder completo con checkpoint disponible en el repositorio.

La prioridad de palabras usa el sistema con menor WER observado. Todos esos modelos
comparten la arquitectura necesaria: frontend profundo, capa de sesión, seis bloques
E-Branchformer, submuestreo 2 y salida de dimensión 384.


## Orden de ejecución

1. Selecciona una **GPU L4** en Colab.
2. Ejecuta la instalación y reinicia la sesión cuando lo indique.
3. Ejecuta de nuevo desde **Configuración** hasta **Zero-shot**.
4. Conserva `EJECUTAR_AJUSTE = False` para hacer solo el experimento principal.
5. Actívalo y vuelve a ejecutar desde Configuración para realizar el ajuste ligero.

El zero-shot no actualiza ningún parámetro. La sección opcional tiene un interruptor
explícito para evitar iniciar entrenamiento y consumir GPU por accidente.


## 0. Instalación

Ejecuta esta celda una vez y después reinicia la sesión de Colab.

In [ ]:
!nvidia-smi

import subprocess, sys, textwrap

def sh(cmd, titulo):
    print(f"  {titulo} ...", end=" ", flush=True)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print("ok" if r.returncode == 0 else "FALLO")
    if r.returncode != 0:
        print(textwrap.indent((r.stdout + r.stderr)[-4000:], "    "))
        raise RuntimeError(f"Fallo: {cmd}")

# Conserva el NumPy de la VM. Es el mismo patrón probado en G01/B01 y evita
# dejar scipy enlazado contra otra ABI durante la instalación de ESPnet.
try:
    import numpy, scipy.special
    numpy_ref = numpy.__version__
except Exception:
    sh(f"{sys.executable} -m pip install -q -U numpy scipy numba",
       "restaurando numpy/scipy")
    import numpy, scipy.special
    numpy_ref = numpy.__version__

constraints = "/content/constraints_j02.txt"
with open(constraints, "w") as f:
    f.write(f"numpy=={numpy_ref}\n")

pip = f"{sys.executable} -m pip install -q -c {constraints}"
sh(f"{pip} 'espnet==202511' espnet_model_zoo loralib h5py pandas matplotlib",
   "ESPnet 202511 y dependencias")

r = subprocess.run(
    [sys.executable, "-c",
     "import torch,numpy,scipy,h5py,loralib,espnet2; "
     "print(torch.__version__, numpy.__version__, torch.cuda.is_available())"],
    capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(r.stderr[-3000:])

print("\nVerificación:", r.stdout.strip())
print("\nINSTALACIÓN TERMINADA. Reinicia la sesión y continúa en la sección 1.")


## 1. Configuración

Los interruptores importantes están reunidos en esta única celda.

In [ ]:
# Rutas de Google Drive
DRIVE_ROOT   = "/content/drive/MyDrive/TFG"
DRIVE_DATA   = f"{DRIVE_ROOT}/DatosEEG_crudos/hdf5_data_final"
DATA_ROOT    = "/content/data/hdf5_data_final"
RESULTS_DIR  = f"{DRIVE_ROOT}/resultados_finales/J02_decoder_owsm"

# Modelo
OWSM_MODEL = "espnet/owsm_v3.1_ebf_base"
LANGUAGE   = "eng"

# Experimento principal
N_ZERO_SHOT          = 200   # muestra uniforme de validación
N_CONTROL_CERO       = 20    # ablación de entrada: estados del encoder a cero
BEAM_SIZE            = 5
INCLUIR_CONTROL_OWSM = True  # añade v5 (encoder iniciado con OWSM), si existe

# Ajuste ligero. False = no se entrena absolutamente nada.
EJECUTAR_AJUSTE          = False
CARGAR_AJUSTE_EXISTENTE  = True
SOBRESCRIBIR_AJUSTE      = False

# Presupuesto del ajuste ligero para una L4
MAX_EPOCHS       = 6
PATIENCE         = 2
CACHE_BATCH_SIZE = 12
TRAIN_BATCH_SIZE = 12
ACCUM_GRAD       = 2
LR_BRIDGE        = 3e-4
LR_LORA          = 1e-4
WEIGHT_DECAY     = 1e-2
LORA_RANK        = 8
LORA_ALPHA       = 16
MAX_TRAIN_TRIALS = None   # None = todos; usa 2000 solo para una prueba rápida

# Datos. Para zero-shot solo se copian los HDF5 de validación.
CACHEAR_HDF5_LOCAL = True
SEED = 2024

print("Modo:", "ZERO-SHOT + AJUSTE" if EJECUTAR_AJUSTE else "SOLO ZERO-SHOT")
print(f"Zero-shot: {N_ZERO_SHOT} trials · beam {BEAM_SIZE}")
if EJECUTAR_AJUSTE:
    print(f"Ajuste: <= {MAX_EPOCHS} épocas · patience {PATIENCE} · encoder congelado")


## 2. Drive y datos

El orden de las sesiones debe ser idéntico al empleado al entrenar la capa de sesión.
Se obtiene ordenando los directorios de Drive, igual que en G01 y B01. Los ficheros se
copian al disco local de la VM porque las lecturas aleatorias sobre Drive son lentas.

Con `EJECUTAR_AJUSTE=False` solo se copia validación. Al activar el ajuste también se
copia entrenamiento.


In [ ]:
import os, re, gc, glob, copy, time, math, random, shutil, hashlib, warnings
from collections import Counter
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

import h5py
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from scipy.ndimage import gaussian_filter1d
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader, Subset

from espnet2.bin.s2t_inference import Speech2Text
from espnet2.layers.create_adapter_fn import create_lora_adapter
from espnet.nets.pytorch_backend.transformer.subsampling import Conv2dSubsampling

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    raise RuntimeError("J02 está preparado para GPU. Activa una L4 en Colab.")

GPU_NAME = torch.cuda.get_device_name(0)
if "L4" not in GPU_NAME:
    warnings.warn(f"Se esperaba una L4; la GPU actual es {GPU_NAME}")

os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

if not os.path.isdir(DRIVE_DATA):
    candidatos = glob.glob(f"{DRIVE_ROOT}/**/hdf5_data_final", recursive=True)
    if not candidatos:
        raise FileNotFoundError(
            f"No encuentro hdf5_data_final bajo {DRIVE_ROOT}. Ajusta DRIVE_DATA.")
    DRIVE_DATA = candidatos[0]
    print("Datos encontrados en:", DRIVE_DATA)

sesiones = []
for nombre in sorted(os.listdir(DRIVE_DATA)):
    carpeta = os.path.join(DRIVE_DATA, nombre)
    if os.path.isdir(carpeta) and os.path.exists(os.path.join(carpeta, "data_train.hdf5")):
        sesiones.append(nombre)
sesiones = sesiones[:45]

if not sesiones:
    raise RuntimeError("No se encontraron sesiones con data_train.hdf5")

print(f"GPU: {GPU_NAME} · sesiones: {len(sesiones)}")
print("Primeras sesiones:", sesiones[:4])

def copiar_split(sesion, split):
    src = os.path.join(DRIVE_DATA, sesion, f"data_{split}.hdf5")
    if not os.path.exists(src):
        return None
    if not CACHEAR_HDF5_LOCAL:
        return src
    dst_dir = os.path.join(DATA_ROOT, sesion)
    dst = os.path.join(dst_dir, f"data_{split}.hdf5")
    os.makedirs(dst_dir, exist_ok=True)
    n_src = os.path.getsize(src)
    if not os.path.exists(dst) or os.path.getsize(dst) != n_src:
        if os.path.exists(dst):
            os.remove(dst)
        shutil.copy2(src, dst)
    try:
        with h5py.File(dst, "r") as f:
            _ = len(f)
    except Exception:
        if os.path.exists(dst):
            os.remove(dst)
        shutil.copy2(src, dst)
        with h5py.File(dst, "r") as f:
            _ = len(f)
    return dst

splits_necesarios = ["val"] + (["train"] if EJECUTAR_AJUSTE else [])
t0 = time.time()
rutas = {split: {} for split in splits_necesarios}
for split in splits_necesarios:
    for i, sesion in enumerate(sesiones, 1):
        ruta = copiar_split(sesion, split)
        if ruta:
            rutas[split][sesion] = ruta
    print(f"  {split}: {len(rutas[split])} ficheros")
print(f"Datos listos en {(time.time()-t0)/60:.1f} min")


In [ ]:
def decode_transcription(arr):
    arr = np.asarray(arr).ravel()
    return "".join(chr(int(x)) for x in arr if int(x) != 0)


def normalizar_texto(texto):
    texto = texto.lower().replace("\x00", "")
    texto = re.sub(r"<[^>]+>", " ", texto)
    texto = re.sub(r"[^a-z0-9' ]+", " ", texto)
    return re.sub(r"\s+", " ", texto).strip()


def construir_indice(split):
    indice = []
    for sid, sesion in enumerate(sesiones):
        ruta = rutas.get(split, {}).get(sesion)
        if not ruta:
            continue
        with h5py.File(ruta, "r") as f:
            for clave in sorted(f.keys()):
                T = int(f[clave]["input_features"].shape[0])
                indice.append((ruta, clave, sid, T))
    return indice


class SeeGIndexDataset(Dataset):
    """Lectura perezosa; mantiene un handle HDF5 por sesión en el proceso principal."""

    def __init__(self, indice):
        self.indice = indice
        self._files = {}

    def __len__(self):
        return len(self.indice)

    def _file(self, path):
        if path not in self._files:
            self._files[path] = h5py.File(path, "r")
        return self._files[path]

    def __getitem__(self, i):
        path, key, sid, _ = self.indice[i]
        g = self._file(path)[key]
        return {
            "features": g["input_features"][:].astype(np.float32),
            "text": decode_transcription(g["transcription"][()]),
            "session_id": sid,
            "trial_id": f"{Path(path).parent.name}/{key}",
        }

    def cerrar(self):
        for f in self._files.values():
            try:
                f.close()
            except Exception:
                pass
        self._files = {}


def preparar_senal(item, suavizado=0.0):
    # Los HDF5 ya vienen normalizados por bloque; reproduce H2/B01/G01.
    x = item["features"].astype(np.float32, copy=False)
    if suavizado > 0:
        x = gaussian_filter1d(x, suavizado, axis=0, mode="nearest").astype(np.float32)
    sid = np.full((x.shape[0], 1), float(item["session_id"]), dtype=np.float32)
    return np.concatenate([x, sid], axis=1)


val_index = construir_indice("val")
val_raw = SeeGIndexDataset(val_index)
train_raw = None
if EJECUTAR_AJUSTE:
    train_index = construir_indice("train")
    train_raw = SeeGIndexDataset(train_index)

print(f"Validación: {len(val_raw):,} trials")
if train_raw is not None:
    print(f"Entrenamiento: {len(train_raw):,} trials")

ejemplo = val_raw[0]
print("Ejemplo:", ejemplo["features"].shape, repr(ejemplo["text"][:90]),
      "sesión", ejemplo["session_id"])


## 3. OWSM y selección del checkpoint

Se descarga OWSM completo para recuperar su decoder y su tokenizador BPE originales.
Después se sustituye únicamente el encoder por el encoder neuronal guardado. La cabeza
CTC del checkpoint se ignora: puede ser de palabras, caracteres o BPE y no forma parte
de esta inferencia.


In [ ]:
# Carpetas creadas por los notebooks finales y por el barrido histórico.
ROOTS_CKPT = [
    f"{DRIVE_ROOT}/resultados_finales",
    f"{DRIVE_ROOT}/resultados",
    f"{DRIVE_ROOT}/proyecto/resultados",
    DRIVE_ROOT,
]

def candidatos_checkpoint(carpeta):
    nombres = ["valid.cer_ctc.best.pth", "valid.loss.best.pth",
               "valid.acc.best.pth", "train.loss.best.pth"]
    return [os.path.join(root, carpeta, nombre)
            for root in ROOTS_CKPT for nombre in nombres]


FUENTES_PRINCIPALES = [
    dict(id="B01_word_final", origen="random", suavizado=0.0,
         carpetas=["B01_word_final"]),
    dict(id="B3word", origen="random", suavizado=0.0,
         carpetas=[
             "B3word_n45_deep_d512b2s2_word_initrandom_sess_ep40_s2024_745c76",
             "B3word_n45_deep_d512b2s2_word_initrandom_sess_ep20_s2024_63ef6e",
         ]),
    dict(id="G01_char_final", origen="random", suavizado=0.0,
         carpetas=["G01_char_final"]),
    dict(id="H2_random", origen="random", suavizado=0.0,
         carpetas=["H2semilla_n45_deep_d512b2s2_char_initrandom_sess_ep40_s1234_191968"]),
]

CONTROL_OWSM = dict(
    id="v5_owsm", origen="owsm", suavizado=2.0,
    carpetas=["v5_n45_deepd512b2s2_ic0.3_augn0.5o0.2_sm2_wd0.01_sess_char_unf1_lora0_lr0.001_els0.05_ctc1_bs16_ep60"],
)

def resolver(spec):
    for carpeta in spec["carpetas"]:
        for path in candidatos_checkpoint(carpeta):
            if os.path.isfile(path) and os.path.getsize(path) > 0:
                return {**spec, "path": path}
    return None


PRIMARY = next((x for x in map(resolver, FUENTES_PRINCIPALES) if x), None)
CONTROL = resolver(CONTROL_OWSM) if INCLUIR_CONTROL_OWSM else None

if PRIMARY is None:
    esperados = []
    for spec in FUENTES_PRINCIPALES:
        esperados.extend(candidatos_checkpoint(spec["carpetas"][0])[:2])
    raise FileNotFoundError(
        "No encuentro ningún checkpoint compatible. Se buscaron, entre otros:\n  "
        + "\n  ".join(esperados)
        + "\nCopia el .pth a una de esas carpetas o ajusta ROOTS_CKPT.")

print("Encoder principal:", PRIMARY["id"])
print("Checkpoint       :", PRIMARY["path"])
print("Control OWSM     :", CONTROL["path"] if CONTROL else "no disponible; se omite")

print("\nDescargando/cargando OWSM...")
s2t = Speech2Text.from_pretrained(
    OWSM_MODEL,
    lang_sym=f"<{LANGUAGE}>", task_sym="<asr>",
    beam_size=BEAM_SIZE, ctc_weight=0.0,
    maxlenratio=0.5, minlenratio=0.0,
    device=DEVICE, dtype="float32", nbest=1,
)
model = s2t.s2t_model
tokenizer = s2t.tokenizer
converter = s2t.converter

ORIGINAL_POS_ENC = copy.deepcopy(model.encoder.embed.out[1]).cpu()
model.frontend = None
model.specaug = None
model.normalize = None

print(f"Decoder: {type(model.decoder).__name__} · vocabulario: {len(converter.token_list):,}")
print(f"Dimensión esperada por el decoder: {model.encoder.output_size()}")


## 4. Reconstrucción exacta del frontend campeón

El frontend y la capa de sesión son los mismos de B01/G01/H2. La carga se hace con
`strict=True` sobre todo el encoder. Si una arquitectura no coincide, el notebook se
detiene en vez de continuar con parámetros sin cargar.


In [ ]:
def _subsample_mask(x_mask, subsample, target_len):
    if x_mask is None:
        return None
    mask = x_mask[:, :, ::subsample]
    if mask.size(2) > target_len:
        mask = mask[:, :, :target_len]
    elif mask.size(2) < target_len:
        mask = torch.nn.functional.pad(mask, (0, target_len-mask.size(2)), value=False)
    return mask


class CapaSesion(nn.Module):
    def __init__(self, n_sesiones, dim=512):
        super().__init__()
        self.n_sesiones = n_sesiones
        self.W = nn.Parameter(torch.eye(dim).unsqueeze(0).repeat(n_sesiones, 1, 1))
        self.b = nn.Parameter(torch.zeros(n_sesiones, dim))

    def forward(self, x):
        sid = x[:, 0, -1].round().long().clamp_(0, self.n_sesiones - 1)
        x = x[:, :, :-1]
        return torch.nn.functional.softsign(
            torch.baddbmm(self.b[sid].unsqueeze(1), x, self.W[sid]))


class BloqueTemporal(nn.Module):
    def __init__(self, dim=512, kernel=5, stride=1, dropout=0.1):
        super().__init__()
        self.conv = nn.Conv1d(dim, dim, kernel, stride=stride, padding=kernel//2)
        self.norm = nn.LayerNorm(dim)
        self.act = nn.GELU()
        self.drop = nn.Dropout(dropout)
        self.residual = stride == 1

    def forward(self, x):
        y = self.conv(x.transpose(1, 2)).transpose(1, 2)
        y = self.drop(self.act(self.norm(y)))
        return x + y if self.residual else y


class FrontendProfundo(Conv2dSubsampling):
    """Frontend de B01/G01: 512 -> 512, temporal, subsample 2, salida 384."""

    def __init__(self, pos_enc, n_sesiones, dim=512, n_res=2,
                 kernel=5, dropout=0.1, subsample=2):
        nn.Module.__init__(self)
        self.sesion = CapaSesion(n_sesiones)
        self.subsample = subsample
        self.espacial = nn.Sequential(
            nn.Linear(512, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(dropout))
        bloques = [BloqueTemporal(dim, kernel, 2, dropout)]
        bloques += [BloqueTemporal(dim, kernel, 1, dropout) for _ in range(n_res)]
        self.bloques = nn.ModuleList(bloques)
        self.salida = nn.Linear(dim, 384)
        self.pos_enc = pos_enc

    def forward(self, x, x_mask):
        x = self.sesion(x)
        x = self.espacial(x)
        if x_mask is not None:
            x = x * x_mask.transpose(1, 2).to(x.dtype)
        for bloque in self.bloques:
            x = bloque(x)
        x = self.pos_enc(self.salida(x))
        x_mask = _subsample_mask(x_mask, self.subsample, x.size(1))
        return x, x_mask


def cargar_torch(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except Exception:
        return torch.load(path, map_location="cpu", weights_only=False)


def normalizar_state_dict(state):
    if isinstance(state, dict) and "model" in state and isinstance(state["model"], dict):
        state = state["model"]
    if any(k.startswith("module.") for k in state):
        state = {k.removeprefix("module."): v for k, v in state.items()}
    return state


def cargar_encoder(spec, verbose=True):
    state = normalizar_state_dict(cargar_torch(spec["path"]))
    enc_state = {k[len("encoder."):]: v for k, v in state.items()
                 if k.startswith("encoder.")}
    if not enc_state:
        raise RuntimeError(f"{spec['path']} no contiene claves encoder.*")

    key_sesion = "embed.sesion.W"
    if key_sesion not in enc_state:
        raise RuntimeError("El checkpoint no contiene la capa de sesión esperada")
    n_ses_ckpt = int(enc_state[key_sesion].shape[0])
    if n_ses_ckpt != len(sesiones):
        raise RuntimeError(
            f"Checkpoint con {n_ses_ckpt} sesiones, datos con {len(sesiones)}. "
            "No se puede reinterpretar la capa de sesión.")

    bloques = sorted({int(m.group(1)) for k in enc_state
                      if (m := re.match(r"encoders\.(\d+)\.", k))})
    if bloques != list(range(6)):
        raise RuntimeError(f"Se esperaban 6 bloques E-Branchformer; aparecen {bloques}")

    pos_enc = copy.deepcopy(ORIGINAL_POS_ENC)
    model.encoder.embed = FrontendProfundo(pos_enc, n_ses_ckpt)
    model.encoder.load_state_dict(enc_state, strict=True)
    model.frontend = model.specaug = model.normalize = None
    model.to(DEVICE).eval()

    for p in model.encoder.parameters():
        p.requires_grad = False
    for p in model.decoder.parameters():
        p.requires_grad = False

    if verbose:
        n_enc = sum(p.numel() for p in model.encoder.parameters())
        print(f"Encoder {spec['id']} cargado estrictamente: {n_enc/1e6:.1f} M parámetros")
        print("  frontend profundo · sesión · subsample 2 · 6 bloques · salida 384")
    del state, enc_state
    gc.collect()
    return model


model = cargar_encoder(PRIMARY)


In [ ]:
# Prueba de forma antes de lanzar una evaluación larga.
item = val_raw[0]
x = torch.from_numpy(preparar_senal(item, PRIMARY["suavizado"])).unsqueeze(0).to(DEVICE)
lens = torch.tensor([x.size(1)], device=DEVICE)

with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
    enc, enc_lens = model.encode(x, lens)
    if isinstance(enc, tuple):
        enc = enc[0]

assert enc.ndim == 3 and enc.shape[-1] == 384
assert int(enc_lens[0]) == enc.shape[1]
print("Señal :", tuple(x.shape))
print("Encoder:", tuple(enc.shape), "· longitud", int(enc_lens[0]))
print("Compatibilidad dimensional con el decoder: OK")

del x, enc
torch.cuda.empty_cache()


## 5. Codificación de la muestra

El encoder se ejecuta una vez por trial. Los estados se guardan en CPU en `float16`;
así las distintas decodificaciones no vuelven a calcular el encoder. En el experimento
zero-shot no se ejecuta ningún `backward()` ni se crea optimizador.


In [ ]:
def indices_uniformes(n_total, n):
    if n is None or n >= n_total:
        return list(range(n_total))
    return np.linspace(0, n_total - 1, n).round().astype(int).tolist()


def collate_senal(items, suavizado):
    arrays = [torch.from_numpy(preparar_senal(x, suavizado)) for x in items]
    lens = torch.tensor([len(x) for x in arrays], dtype=torch.long)
    speech = pad_sequence(arrays, batch_first=True, padding_value=0.0)
    return speech, lens, [x["text"] for x in items], [x["trial_id"] for x in items]


@torch.inference_mode()
def codificar_subset(dataset, indices, spec, batch_size=CACHE_BATCH_SIZE):
    cargar_encoder(spec)
    subset = Subset(dataset, indices)
    loader = DataLoader(
        subset, batch_size=batch_size, shuffle=False, num_workers=0,
        pin_memory=True,
        collate_fn=lambda xs: collate_senal(xs, spec["suavizado"]),
    )

    estados, refs, ids = [], [], []
    t0 = time.time()
    for paso, (speech, lens, textos, trial_ids) in enumerate(loader, 1):
        speech = speech.to(DEVICE, non_blocking=True)
        lens = lens.to(DEVICE, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16):
            enc, out_lens = model.encode(speech, lens)
            if isinstance(enc, tuple):
                enc = enc[0]
        for b, longitud in enumerate(out_lens.tolist()):
            estados.append(enc[b, :longitud].detach().cpu().to(torch.float16))
        refs.extend(textos)
        ids.extend(trial_ids)
        if paso % max(1, len(loader)//10) == 0 or paso == len(loader):
            print(f"\r  codificando {len(estados):,}/{len(subset):,}", end="")
    print(f" · {(time.time()-t0)/60:.1f} min")
    mb = sum(x.numel() * x.element_size() for x in estados) / 2**20
    print(f"  caché de estados: {mb:.0f} MiB en CPU")
    return {"states": estados, "refs": refs, "trial_ids": ids,
            "indices": indices, "source": spec["id"]}


## 6. Inferencia zero-shot: decoder de atención puro

La búsqueda usa pesos `decoder=1` y `ctc=0`. Esto es obligatorio porque la cabeza CTC
del encoder seleccionado y el decoder OWSM no comparten vocabulario. El decoder y sus
50.002 salidas conservan exactamente los pesos preentrenados.

Además se decodifican veinte entradas nulas. Si la salida apenas cambia al borrar toda
la información del encoder, el decoder está actuando principalmente como prior de
lenguaje.


In [ ]:
def rebind_beam_search():
    """Apunta todos los scorers al decoder actual; patrón probado en los notebooks previos."""
    s2t.s2t_model = model
    beam = s2t.beam_search
    for scorers in (getattr(beam, "scorers", {}),
                    getattr(beam, "full_scorers", {}),
                    getattr(beam, "part_scorers", {})):
        if "decoder" in scorers:
            scorers["decoder"] = model.decoder
    if hasattr(beam, "nn_dict") and "decoder" in beam.nn_dict:
        beam.nn_dict["decoder"] = model.decoder
    if hasattr(beam, "weights"):
        beam.weights["ctc"] = 0.0
        beam.weights["decoder"] = 1.0
    beam.to(device=DEVICE).eval()
    assert abs(float(beam.weights.get("ctc", 0.0))) < 1e-12
    assert abs(float(beam.weights["decoder"]) - 1.0) < 1e-12


def distancia(a, b):
    dp = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev, dp[0] = dp[0], i
        for j, cb in enumerate(b, 1):
            old = dp[j]
            dp[j] = min(dp[j] + 1, dp[j-1] + 1, prev + (ca != cb))
            prev = old
    return dp[-1]


def error_trial(ref, hyp, unidad="word"):
    r, h = normalizar_texto(ref), normalizar_texto(hyp)
    r = r.split() if unidad == "word" else list(r)
    h = h.split() if unidad == "word" else list(h)
    return 0.0 if not r else distancia(r, h) / len(r)


def metricas_corpus(refs, hyps):
    wers = [error_trial(r, h, "word") for r, h in zip(refs, hyps)]
    cers = [error_trial(r, h, "char") for r, h in zip(refs, hyps)]
    wer_num = sum(distancia(normalizar_texto(r).split(), normalizar_texto(h).split())
                  for r, h in zip(refs, hyps))
    wer_den = sum(len(normalizar_texto(r).split()) for r in refs)
    cer_num = sum(distancia(list(normalizar_texto(r)), list(normalizar_texto(h)))
                  for r, h in zip(refs, hyps))
    cer_den = sum(len(normalizar_texto(r)) for r in refs)
    return dict(
        wer_macro=float(np.mean(wers)), wer_micro=wer_num/max(1, wer_den),
        cer_macro=float(np.mean(cers)), cer_micro=cer_num/max(1, cer_den),
        hipotesis_vacias=float(np.mean([not normalizar_texto(h) for h in hyps])),
        hipotesis_distintas=len(set(map(normalizar_texto, hyps))),
        moda_frac=Counter(map(normalizar_texto, hyps)).most_common(1)[0][1] / len(hyps),
    )


@torch.inference_mode()
def decodificar_estado(state, bridge=None, anular=False):
    rebind_beam_search()
    model.decoder.eval()
    memory = state.to(DEVICE, dtype=torch.float32)
    if anular:
        memory = torch.zeros_like(memory)
    if bridge is not None:
        bridge.eval()
        memory = bridge(memory.unsqueeze(0))[0]

    lang_id = s2t.converter.token2id[f"<{LANGUAGE}>"]
    task_id = s2t.converter.token2id["<asr>"]
    notime_id = s2t.converter.token2id[s2t.preprocessor_conf["notime_symbol"]]
    s2t.beam_search.set_hyp_primer([model.sos, lang_id, task_id, notime_id])
    resultado = s2t._decode_single_sample(memory)[0]
    text, _, _, text_nospecial, _ = resultado
    return (text_nospecial or text or "").strip()


def evaluar_cache(cache, etiqueta, bridge=None, anular=False, n=None):
    n = len(cache["states"]) if n is None else min(n, len(cache["states"]))
    refs, hyps = cache["refs"][:n], []
    t0 = time.time()
    nivel = __import__("logging").getLogger().level
    __import__("logging").getLogger().setLevel(__import__("logging").WARNING)
    try:
        for i, state in enumerate(cache["states"][:n], 1):
            hyps.append(decodificar_estado(state, bridge=bridge, anular=anular))
            if i % max(1, n//20) == 0 or i == n:
                print(f"\r  {etiqueta}: {i}/{n}", end="")
    finally:
        __import__("logging").getLogger().setLevel(nivel)
    print(f" · {(time.time()-t0)/60:.1f} min")

    met = metricas_corpus(refs, hyps)
    filas = pd.DataFrame({
        "trial_id": cache["trial_ids"][:n], "referencia": refs, "hipotesis": hyps,
        "wer": [error_trial(r, h, "word") for r, h in zip(refs, hyps)],
        "cer": [error_trial(r, h, "char") for r, h in zip(refs, hyps)],
    })
    return filas, met


print("Beam search preparado: decoder=1.0 · CTC=0.0")


In [ ]:
specs_zero = [PRIMARY] + ([CONTROL] if CONTROL is not None else [])
resumen_zero, predicciones_zero = [], {}
PRIMARY_ZERO_CACHE = None

for spec in specs_zero:
    print("\n" + "="*76)
    print(f"ZERO-SHOT · {spec['id']} · origen={spec['origen']}")
    print("="*76)
    idx = indices_uniformes(len(val_raw), N_ZERO_SHOT)
    cache = codificar_subset(val_raw, idx, spec)
    if spec["id"] == PRIMARY["id"]:
        PRIMARY_ZERO_CACHE = cache

    pred, met = evaluar_cache(cache, f"decoder {spec['id']}")
    pred["modelo"] = spec["id"]
    pred["metodo"] = "zero_shot"

    # Control sin señal, sobre exactamente los primeros trials de la muestra.
    pred_cero, met_cero = evaluar_cache(
        cache, f"entrada cero {spec['id']}", anular=True, n=N_CONTROL_CERO)
    iguales_cero = np.mean([
        normalizar_texto(a) == normalizar_texto(b)
        for a, b in zip(pred["hipotesis"][:len(pred_cero)], pred_cero["hipotesis"])
    ])

    # Control de emparejamiento sin repetir la decodificación: rota las referencias.
    refs_rotadas = list(pred["referencia"].iloc[1:]) + [pred["referencia"].iloc[0]]
    wer_perm = metricas_corpus(refs_rotadas, pred["hipotesis"].tolist())["wer_micro"]

    fila = dict(
        modelo=spec["id"], origen_encoder=spec["origen"], metodo="zero_shot",
        n=len(pred), **met,
        wer_micro_referencias_rotadas=wer_perm,
        fraccion_igual_entrada_cero=iguales_cero,
    )
    resumen_zero.append(fila)
    predicciones_zero[spec["id"]] = pred
    pred.to_csv(f"{RESULTS_DIR}/predicciones_zero_shot_{spec['id']}.csv", index=False)

    print(f"  WER micro: {met['wer_micro']:.3f} · WER macro: {met['wer_macro']:.3f}")
    print(f"  CER micro: {met['cer_micro']:.3f} · hipótesis distintas: "
          f"{met['hipotesis_distintas']}/{len(pred)}")
    print(f"  moda: {met['moda_frac']:.1%} · vacías: {met['hipotesis_vacias']:.1%}")
    print(f"  mismas salidas al anular encoder: {iguales_cero:.1%} (n={len(pred_cero)})")
    display(pred[["referencia", "hipotesis", "wer"]].head(8))

df_zero = pd.DataFrame(resumen_zero)
df_zero.to_csv(f"{RESULTS_DIR}/resumen_zero_shot.csv", index=False)
display(df_zero.T)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
nombres = df_zero["modelo"]
x = np.arange(len(df_zero))
ax[0].bar(x-0.18, df_zero["wer_micro"], width=.36, label="WER micro", color="#2B6CB0")
ax[0].bar(x+0.18, df_zero["wer_macro"], width=.36, label="WER medio/trial", color="#D97706")
ax[0].set_xticks(x, nombres, rotation=12, ha="right")
ax[0].set_ylabel("WER (menor es mejor)")
ax[0].set_title("Decoder OWSM sin reentrenamiento")
ax[0].grid(axis="y", alpha=.25)
ax[0].legend()

ax[1].bar(x-0.18, df_zero["moda_frac"], width=.36, label="fracción de la moda", color="#6B7280")
ax[1].bar(x+0.18, df_zero["fraccion_igual_entrada_cero"], width=.36,
          label="igual con entrada nula", color="#B91C1C")
ax[1].set_xticks(x, nombres, rotation=12, ha="right")
ax[1].set_ylim(0, 1)
ax[1].set_ylabel("Fracción")
ax[1].set_title("¿Usa el decoder la señal del encoder?")
ax[1].grid(axis="y", alpha=.25)
ax[1].legend()

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/fig_zero_shot_decoder_owsm.png", dpi=180, bbox_inches="tight")
plt.show()
print("Guardado en:", RESULTS_DIR)


### Lectura del zero-shot

- Un WER alto **no prueba que el decoder de OWSM sea inútil**. Prueba que el encoder
  CTC y el decoder no son directamente compatibles bajo este ensamblaje sin ajuste.
- Muchas hipótesis repetidas, o salidas iguales al anular el encoder, indican que el
  decoder está usando principalmente su prior lingüístico.
- Si `v5_owsm` supera al encoder principal, pese a tener peor CTC, sería evidencia de
  que la inicialización OWSM conservó parte de la geometría esperada por el decoder.
- Si el encoder principal gana, la calidad aprendida para sEEG pesa más que esa posible
  conservación de geometría.

La conclusión principal debe apoyarse en el WER y también en los controles de diversidad
y entrada nula; una frase plausible aislada no es evidencia de transferencia.


## 7. Ajuste ligero opcional

Esta sección **no reentrena el encoder**. Primero almacena sus estados una sola vez y lo
saca de la GPU. Después entrena:

```text
estados sEEG congelados
        -> LayerNorm + MLP residual 384 -> 768 -> 384
        -> decoder OWSM congelado con LoRA solo en cross-attention
        -> BPE original de OWSM
```

La pérdida es la entropía cruzada autorregresiva original de OWSM mediante
`model._calc_att_loss`. No se usa CTC en esta fase. Esto mantiene la pregunta limpia:
cuánta alineación mínima necesita el decoder para aprovechar un encoder ya aprendido.

Activa `EJECUTAR_AJUSTE=True` en la configuración y vuelve a ejecutar desde allí. La
selección del checkpoint se hace por pérdida de validación; el WER se calcula una sola
vez al final para limitar el coste autorregresivo.


In [ ]:
CACHE_DIR = "/content/j02_cache"
os.makedirs(CACHE_DIR, exist_ok=True)
ADAPT_CKPT = f"{RESULTS_DIR}/bridge_lora_best.pth"

TRAIN_CACHE = VALID_CACHE = None

def fingerprint(spec):
    st = os.stat(spec["path"])
    raw = f"{spec['id']}|{st.st_size}|{st.st_mtime_ns}".encode()
    return hashlib.sha1(raw).hexdigest()[:12]


def cargar_o_crear_cache(dataset, split, spec, max_items=None):
    fp = fingerprint(spec)
    path = f"{CACHE_DIR}/{spec['id']}_{split}_{fp}.pth"
    if os.path.exists(path):
        print("Cargando caché:", path)
        return cargar_torch(path)
    indices = indices_uniformes(len(dataset), max_items)
    cache = codificar_subset(dataset, indices, spec)
    torch.save(cache, path)
    print("Caché guardada:", path)
    return cache


if EJECUTAR_AJUSTE:
    if train_raw is None:
        raise RuntimeError(
            "train_raw no existe. Activa EJECUTAR_AJUSTE en la configuración y "
            "vuelve a ejecutar las secciones 1 y 2 para copiar/indexar train.")
    print("\nPrecálculo del encoder principal. No hay gradientes.")
    TRAIN_CACHE = cargar_o_crear_cache(
        train_raw, "train", PRIMARY, max_items=MAX_TRAIN_TRIALS)
    VALID_CACHE = cargar_o_crear_cache(val_raw, "valid", PRIMARY, max_items=None)
    model.encoder.cpu()
    torch.cuda.empty_cache()
    print("Encoder retirado de la GPU. El entrenamiento usará solo estados guardados.")
else:
    print("Ajuste desactivado: no se codifica train y no se entrena nada.")


In [ ]:
class PuenteResidual(nn.Module):
    """Identidad al inicio; aprende solo la corrección necesaria para el decoder."""

    def __init__(self, dim=384, hidden=768, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.net = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, dim),
        )
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, x):
        return x + self.net(self.norm(x))


def preparar_lora_decoder():
    # Congela todos los pesos base antes de insertar LoRA.
    for p in model.parameters():
        p.requires_grad = False

    suffixes = ("src_attn.linear_q", "src_attn.linear_k",
                "src_attn.linear_v", "src_attn.linear_out")
    targets = [name for name, module in model.decoder.named_modules()
               if isinstance(module, nn.Linear) and name.endswith(suffixes)]
    if len(targets) != 24:
        raise RuntimeError(
            f"Se esperaban 24 proyecciones de cross-attention (6x4); hay {len(targets)}: {targets}")

    create_lora_adapter(
        model.decoder, rank=LORA_RANK, alpha=LORA_ALPHA,
        dropout_rate=0.05, target_modules=targets, bias_type="none")

    for name, p in model.decoder.named_parameters():
        p.requires_grad = "lora_" in name
    return targets


LORA_READY = False
bridge = None

if EJECUTAR_AJUSTE or (CARGAR_AJUSTE_EXISTENTE and os.path.exists(ADAPT_CKPT)):
    targets_lora = preparar_lora_decoder()
    LORA_READY = True
    bridge = PuenteResidual().to(DEVICE)
    model.decoder.to(DEVICE)

    n_bridge = sum(p.numel() for p in bridge.parameters() if p.requires_grad)
    n_lora = sum(p.numel() for p in model.decoder.parameters() if p.requires_grad)
    n_decoder = sum(p.numel() for p in model.decoder.parameters())
    print(f"Puente entrenable: {n_bridge:,} parámetros")
    print(f"LoRA entrenable  : {n_lora:,} / {n_decoder:,} parámetros del decoder")
    print(f"Total entrenable : {n_bridge+n_lora:,}")
    print("Encoder entrenable: 0")
else:
    print("No se crean adaptadores: termina aquí el recorrido exclusivamente zero-shot.")


In [ ]:
class CachedTextDataset(Dataset):
    def __init__(self, cache):
        self.states = cache["states"]
        self.refs = cache["refs"]
        self.text_ids = []
        self.prev_ids = []
        prev = np.asarray(converter.tokens2ids(tokenizer.text2tokens("<na>")), dtype=np.int64)
        for ref in self.refs:
            objetivo = f"<{LANGUAGE}><asr><notimestamps> {ref.lower()}"
            ids = converter.tokens2ids(tokenizer.text2tokens(objetivo))
            self.text_ids.append(torch.tensor(ids, dtype=torch.long))
            self.prev_ids.append(torch.tensor(prev, dtype=torch.long))

    def __len__(self):
        return len(self.states)

    def __getitem__(self, i):
        return self.states[i], self.text_ids[i], self.prev_ids[i]


class SortedBatchSampler:
    """Lotes de longitudes parecidas; baraja el orden de los lotes cada época."""

    def __init__(self, dataset, batch_size, shuffle=True):
        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle

    def __len__(self):
        return math.ceil(len(self.dataset) / self.batch_size)

    def __iter__(self):
        indices = sorted(range(len(self.dataset)), key=lambda i: len(self.dataset.states[i]))
        batches = [indices[i:i+self.batch_size]
                   for i in range(0, len(indices), self.batch_size)]
        if self.shuffle:
            random.shuffle(batches)
        yield from batches


def collate_estados(items):
    states, texts, prevs = zip(*items)
    state_lens = torch.tensor([len(x) for x in states], dtype=torch.long)
    text_lens = torch.tensor([len(x) for x in texts], dtype=torch.long)
    prev_lens = torch.tensor([len(x) for x in prevs], dtype=torch.long)
    return (
        pad_sequence(states, batch_first=True, padding_value=0.0), state_lens,
        pad_sequence(texts, batch_first=True, padding_value=model.ignore_id), text_lens,
        pad_sequence(prevs, batch_first=True, padding_value=model.ignore_id), prev_lens,
    )


def mover_batch(batch):
    return tuple(x.to(DEVICE, non_blocking=True) for x in batch)


def perdida_decoder(batch):
    states, state_lens, text, text_lens, prev, prev_lens = mover_batch(batch)
    memory = bridge(states)
    loss, acc, _, _ = model._calc_att_loss(
        memory, state_lens, text, text_lens, prev, prev_lens)
    return loss, acc


def estado_entrenable():
    return {
        "bridge": {k: v.detach().cpu().clone() for k, v in bridge.state_dict().items()},
        "lora": {k: v.detach().cpu().clone()
                 for k, v in model.decoder.state_dict().items() if "lora_" in k},
    }


def cargar_estado_entrenable(payload):
    # train() deshace el merge de LoRA antes de sustituir A/B.
    model.decoder.train()
    bridge.load_state_dict(payload["bridge"], strict=True)
    faltan = model.decoder.load_state_dict(payload["lora"], strict=False)
    inesperadas = [k for k in faltan.unexpected_keys if "lora_" in k]
    if inesperadas:
        raise RuntimeError(f"Claves LoRA inesperadas: {inesperadas}")
    model.decoder.eval()
    bridge.eval()


In [ ]:
def crear_grad_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=True)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=True)


def entrenar_una_epoca(loader, optimizer, scheduler, scaler):
    model.train()                    # evita métricas textuales dentro de _calc_att_loss
    model.decoder.train()            # LoRA sin fusionar, dropout activo
    bridge.train()
    optimizer.zero_grad(set_to_none=True)
    total_loss = total_acc = total_n = 0.0

    for paso, batch in enumerate(loader, 1):
        with torch.autocast("cuda", dtype=torch.float16):
            loss, acc = perdida_decoder(batch)
            loss_back = loss / ACCUM_GRAD
        scaler.scale(loss_back).backward()

        if paso % ACCUM_GRAD == 0 or paso == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                list(bridge.parameters()) +
                [p for p in model.decoder.parameters() if p.requires_grad], 5.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        n = batch[0].size(0)
        total_loss += float(loss.detach()) * n
        total_acc += float(acc) * n
        total_n += n
        if paso % max(1, len(loader)//10) == 0 or paso == len(loader):
            print(f"\r    train {paso}/{len(loader)} · loss {total_loss/total_n:.3f}", end="")
    print()
    return total_loss/total_n, total_acc/total_n


@torch.no_grad()
def validar(loader):
    # model.training=True evita calcular CER/WER teacher-forced; el decoder se deja
    # en eval para quitar dropout y fusionar LoRA durante esta pasada.
    model.train()
    model.decoder.eval()
    bridge.eval()
    total_loss = total_acc = total_n = 0.0
    for batch in loader:
        with torch.autocast("cuda", dtype=torch.float16):
            loss, acc = perdida_decoder(batch)
        n = batch[0].size(0)
        total_loss += float(loss) * n
        total_acc += float(acc) * n
        total_n += n
    return total_loss/total_n, total_acc/total_n


def ejecutar_ajuste():
    train_ds = CachedTextDataset(TRAIN_CACHE)
    valid_ds = CachedTextDataset(VALID_CACHE)
    train_loader = DataLoader(
        train_ds, batch_sampler=SortedBatchSampler(train_ds, TRAIN_BATCH_SIZE, True),
        collate_fn=collate_estados, num_workers=0, pin_memory=True)
    valid_loader = DataLoader(
        valid_ds, batch_sampler=SortedBatchSampler(valid_ds, TRAIN_BATCH_SIZE, False),
        collate_fn=collate_estados, num_workers=0, pin_memory=True)

    lora_params = [p for p in model.decoder.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW([
        {"params": bridge.parameters(), "lr": LR_BRIDGE},
        {"params": lora_params, "lr": LR_LORA},
    ], weight_decay=WEIGHT_DECAY)

    updates_epoch = math.ceil(len(train_loader) / ACCUM_GRAD)
    total_updates = max(1, updates_epoch * MAX_EPOCHS)
    warmup = min(300, max(20, updates_epoch//2))

    def lr_factor(step):
        if step < warmup:
            return max(1e-3, (step + 1) / warmup)
        progress = (step - warmup) / max(1, total_updates - warmup)
        return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1.0, progress)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
    scaler = crear_grad_scaler()

    historia, best_loss, sin_mejora = [], float("inf"), 0
    print(f"Train {len(train_ds):,} · valid {len(valid_ds):,} · "
          f"{updates_epoch} updates/época · warmup {warmup}")

    for epoch in range(1, MAX_EPOCHS + 1):
        t0 = time.time()
        train_loss, train_acc = entrenar_una_epoca(
            train_loader, optimizer, scheduler, scaler)
        valid_loss, valid_acc = validar(valid_loader)
        minutos = (time.time() - t0) / 60
        fila = dict(epoch=epoch, train_loss=train_loss, valid_loss=valid_loss,
                    train_acc=train_acc, valid_acc=valid_acc,
                    lr_bridge=optimizer.param_groups[0]["lr"],
                    lr_lora=optimizer.param_groups[1]["lr"], minutos=minutos)
        historia.append(fila)
        pd.DataFrame(historia).to_csv(f"{RESULTS_DIR}/historia_ajuste.csv", index=False)
        print(f"  época {epoch}: train {train_loss:.3f}/{train_acc:.3f} · "
              f"valid {valid_loss:.3f}/{valid_acc:.3f} · {minutos:.1f} min")

        if valid_loss < best_loss - 1e-4:
            best_loss, sin_mejora = valid_loss, 0
            estado = estado_entrenable()
            payload = {
                **estado, "source_id": PRIMARY["id"],
                "source_fingerprint": fingerprint(PRIMARY),
                "epoch": epoch, "valid_loss": valid_loss,
                "config": dict(lora_rank=LORA_RANK, lora_alpha=LORA_ALPHA,
                               lr_bridge=LR_BRIDGE, lr_lora=LR_LORA,
                               train_batch=TRAIN_BATCH_SIZE, accum_grad=ACCUM_GRAD),
            }
            torch.save(payload, ADAPT_CKPT)
            print("    nuevo mejor checkpoint")
        else:
            sin_mejora += 1
            if sin_mejora >= PATIENCE:
                print(f"  early stopping: {PATIENCE} épocas sin mejora")
                break

    payload = cargar_torch(ADAPT_CKPT)
    cargar_estado_entrenable(payload)
    print(f"Mejor ajuste: época {payload['epoch']} · valid loss {payload['valid_loss']:.4f}")
    return pd.DataFrame(historia)


In [ ]:
AJUSTE_DISPONIBLE = False

if LORA_READY:
    existe = os.path.exists(ADAPT_CKPT)
    if existe:
        previo = cargar_torch(ADAPT_CKPT)
        mismo_encoder = (previo.get("source_id") == PRIMARY["id"] and
                          previo.get("source_fingerprint") == fingerprint(PRIMARY))
        if not mismo_encoder:
            raise RuntimeError(
                "El ajuste guardado pertenece a otro encoder. Cambia RESULTS_DIR o "
                "activa SOBRESCRIBIR_AJUSTE tras revisar el checkpoint.")

    if EJECUTAR_AJUSTE and (SOBRESCRIBIR_AJUSTE or not existe):
        HISTORIA = ejecutar_ajuste()
        AJUSTE_DISPONIBLE = True
    elif existe and CARGAR_AJUSTE_EXISTENTE:
        payload = cargar_torch(ADAPT_CKPT)
        cargar_estado_entrenable(payload)
        AJUSTE_DISPONIBLE = True
        print(f"Ajuste existente cargado: época {payload['epoch']} · "
              f"valid loss {payload['valid_loss']:.4f}")
    elif EJECUTAR_AJUSTE and existe:
        print("Ya existe un ajuste. Activa SOBRESCRIBIR_AJUSTE para repetirlo.")
    else:
        print("No se carga ni se entrena el ajuste.")
else:
    print("Recorrido zero-shot completado; ajuste no solicitado.")


## 8. Evaluación final del ajuste

Se usa exactamente la misma muestra y el mismo beam que en zero-shot. Por tanto, la
diferencia de WER se debe al puente y a LoRA, no a un cambio de ejemplos o búsqueda.


In [ ]:
if AJUSTE_DISPONIBLE:
    pred_adapt, met_adapt = evaluar_cache(
        PRIMARY_ZERO_CACHE, "decoder ajustado", bridge=bridge)
    pred_adapt["modelo"] = PRIMARY["id"]
    pred_adapt["metodo"] = "bridge_lora"
    pred_adapt.to_csv(f"{RESULTS_DIR}/predicciones_bridge_lora.csv", index=False)

    zero_primary = df_zero[df_zero["modelo"] == PRIMARY["id"]].iloc[0].to_dict()
    comparacion = pd.DataFrame([
        dict(metodo="zero_shot", wer_micro=zero_primary["wer_micro"],
             wer_macro=zero_primary["wer_macro"], cer_micro=zero_primary["cer_micro"],
             cer_macro=zero_primary["cer_macro"]),
        dict(metodo="bridge_lora", wer_micro=met_adapt["wer_micro"],
             wer_macro=met_adapt["wer_macro"], cer_micro=met_adapt["cer_micro"],
             cer_macro=met_adapt["cer_macro"]),
    ])
    comparacion["delta_wer_micro_vs_zero"] = (
        comparacion["wer_micro"] - float(zero_primary["wer_micro"]))
    comparacion.to_csv(f"{RESULTS_DIR}/comparacion_zero_vs_ajuste.csv", index=False)
    display(comparacion)
    display(pred_adapt[["referencia", "hipotesis", "wer"]].head(10))

    fig, ax = plt.subplots(figsize=(7, 4))
    x = np.arange(2)
    ax.bar(x-0.18, comparacion["wer_micro"], .36, label="WER micro", color="#2B6CB0")
    ax.bar(x+0.18, comparacion["wer_macro"], .36, label="WER medio/trial", color="#D97706")
    ax.set_xticks(x, ["zero-shot", "puente + LoRA"])
    ax.set_ylabel("WER (menor es mejor)")
    ax.set_title(f"Reutilización del decoder OWSM · {PRIMARY['id']}")
    ax.grid(axis="y", alpha=.25)
    ax.legend()
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/fig_zero_vs_ajuste.png", dpi=180, bbox_inches="tight")
    plt.show()
else:
    print("Sin ajuste cargado: la conclusión disponible es exclusivamente zero-shot.")


## 9. Conclusión reproducible

La celda siguiente redacta un resumen descriptivo a partir de las cifras. La redacción
final de la memoria debe mantener el alcance exacto:

- **Zero-shot malo:** no existe compatibilidad directa demostrable entre el espacio
  aprendido por el encoder CTC y el decoder OWSM preentrenado.
- **Mejora con puente + LoRA:** el conocimiento del decoder es reutilizable, pero exige
  alineación supervisada; no es transferencia zero-shot.
- **Sin mejora tras el ajuste:** este mecanismo ligero no basta. No demuestra que una
  adaptación conjunta como BrainWhisperer sea imposible.

Este experimento usa validación de las mismas sesiones y, si se ajusta, selecciona por
pérdida de esa validación. No sustituye a un test ciego ni reproduce BrainWhisperer:
allí se optimizan conjuntamente objetivos fonéticos y textuales y se modifica el encoder.


In [ ]:
principal = df_zero[df_zero["modelo"] == PRIMARY["id"]].iloc[0]

print("CONCLUSIÓN DESCRIPTIVA DE J02")
print("-" * 72)
print(f"Encoder principal: {PRIMARY['id']} ({PRIMARY['origen']})")
print(f"Zero-shot: WER micro {principal.wer_micro:.3f}; "
      f"WER medio/trial {principal.wer_macro:.3f}; "
      f"{int(principal.hipotesis_distintas)}/{int(principal.n)} hipótesis distintas.")
print(f"La salida coincide con la obtenida al anular el encoder en "
      f"{principal.fraccion_igual_entrada_cero:.1%} del control.")

if AJUSTE_DISPONIBLE:
    delta = met_adapt["wer_micro"] - principal.wer_micro
    print(f"Puente + LoRA: WER micro {met_adapt['wer_micro']:.3f}; "
          f"cambio frente a zero-shot {delta:+.3f}.")
    if delta < -0.02:
        print("Lectura: hay señal de que el decoder preentrenado es reutilizable, "
              "pero necesita alineación supervisada.")
    elif abs(delta) <= 0.02:
        print("Lectura: el ajuste ligero no produce una mejora clara en esta muestra.")
    else:
        print("Lectura: el ajuste ligero empeora el WER; no logra alinear ambos espacios.")
else:
    print("No se ejecutó ajuste. J02 responde únicamente a la compatibilidad directa zero-shot.")

print(f"\nArtefactos: {RESULTS_DIR}")


---

## Notas metodológicas para la memoria

1. `WER micro` es el WER estándar de corpus: suma ediciones y divide entre todas las
   palabras de referencia. `WER medio/trial` mantiene la convención de varios notebooks
   anteriores y se presenta para trazabilidad.
2. El zero-shot conserva intactos decoder, embeddings y proyección de salida de OWSM.
   Solo se sustituye el encoder acústico por el encoder neuronal.
3. La inferencia es de atención pura. Combinarla con CTC sería inválido cuando CTC usa
   caracteres o palabras y el decoder usa el BPE de OWSM.
4. En el ajuste, el encoder tiene cero parámetros entrenables y se ejecuta una sola vez.
5. El puente se inicializa como identidad. LoRA se limita a las 24 proyecciones de
   atención cruzada (`6 capas x {q,k,v,out}`), que son la interfaz directa con el encoder.
6. No se debe seleccionar ejemplos cualitativos a mano. Los CSV guardan las 200
   predicciones y la muestra es uniforme y determinista.
